# CareLoop Clinics — No-Show Guard
## Data Pipeline: Ingestion, Validation, Cleaning, and Weather Enrichment

**Author:** Tarun Shahi  
**Project:** CareLoop Clinics Patient No-Show Analytics & Early Risk Prediction  
**Input Data:** `data/appointments.csv`, `data/clinics.csv`  
**Outputs Generated:** `rejected_rows.json`, `appointments_clean.csv`, `clinics_enriched.csv`, `load_data.sql`

---

### Overview
CareLoop Clinics operates eight outpatient centres across six Indian cities. Unattended appointments leave doctor slots empty that could have served other patients. Currently, clinic front desks only identify no-shows at the end of the day.

This notebook parses raw booking exports, validates data types, handles missing values and keying errors, enriches clinic records with weather data from Open-Meteo, and generates cleaned files for downstream modeling and SQL analysis.


## 1. Raw Ingestion & Schema Validation

Parse `data/appointments.csv` using Python's standard `csv` module, casting numeric columns (`age`, `fee_inr`, `prior_no_shows` to `int`, and `distance_km` to `float`). Rows that fail type conversion are routed to `rejected_rows.json`, while valid rows are retained in an accepted list.


In [1]:
import os
from pathlib import Path

def resolve_input_path(path_str: str) -> str:
    for candidate in [Path(path_str), Path("..") / path_str, Path("../data") / Path(path_str).name, Path("data") / Path(path_str).name]:
        if candidate.exists():
            return str(candidate)
    return path_str

def get_output_dir(dir_name: str) -> Path:
    target = Path(dir_name) if Path(dir_name).exists() else Path("..") / dir_name
    target.mkdir(parents=True, exist_ok=True)
    return target

# Locate input raw datasets
APPOINTMENTS_PATH = resolve_input_path("data/appointments.csv")
CLINICS_PATH = resolve_input_path("data/clinics.csv")
PROCESSED_DIR = Path(".")
SQL_DIR = Path(".")

import csv
import json

# Open raw appointments file with csv.DictReader
with open(APPOINTMENTS_PATH, mode="r", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames
    raw_rows = list(reader)

print(f"Header: {header}")
print(f"Raw row count: {len(raw_rows):,}")

def parse_row(row: dict) -> dict:
    """
    Converts numeric fields to target primitive types:
    - age, fee_inr, prior_no_shows -> int
    - distance_km -> float
    Raises ValueError if conversion fails due to empty strings or non-numeric characters.
    """
    converted = dict(row)
    converted["age"] = int(row["age"])
    converted["fee_inr"] = int(row["fee_inr"])
    converted["prior_no_shows"] = int(row["prior_no_shows"])
    converted["distance_km"] = float(row["distance_km"])
    return converted

accepted = []
rejected = []

for row in raw_rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)

print(f"Accepted rows: {len(accepted):,}")
print(f"Rejected rows: {len(rejected):,}")

# Save rejected rows with indentation
with open("rejected_rows.json", mode="w", encoding="utf-8") as f:
    json.dump(rejected, f, indent=2)

# Inspect rejection reasons
blank_age = sum(1 for r in rejected if r["age"] == "")
blank_distance = sum(1 for r in rejected if r["distance_km"] == "")
print(f"  - Rejected due to blank age: {blank_age}")
print(f"  - Rejected due to blank distance_km: {blank_distance}")

# Verify accepted rows with impossible ages (validity vs type check)
impossible_ages = [r for r in accepted if int(r["age"]) < 1 or int(r["age"]) > 100]
print(f"  - Accepted rows with out-of-range ages: {len(impossible_ages)}")


Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Raw row count: 1,020
Accepted rows: 988
Rejected rows: 32
  - Rejected due to blank age: 12
  - Rejected due to blank distance_km: 20
  - Accepted rows with impossible ages (e.g. >100): 5 (accepted because type check != validity check)


## 2. Deduplication & String Normalization

Load the raw appointments dataset with pandas to perform initial cleaning:
1. Remove exact duplicate records.
2. Standardize `booking_channel` strings (`.str.strip().str.title()`).
3. Parse date fields (`appt_date` with `errors="coerce"` and `booked_on`).
4. Drop rows with invalid dates or missing appointment status.


In [2]:
import pandas as pd
import numpy as np

# Load fresh copy of raw appointments
df = pd.read_csv(APPOINTMENTS_PATH)
print(f"Raw shape: {df.shape}")
print("\nMissing values count per column:")
print(df.isnull().sum())
print(f"\nDuplicate rows count: {df.duplicated().sum()}")
print(f"Distinct booking_channel values before cleaning: {df['booking_channel'].nunique()}")

# (a) Drop exact duplicate rows
df = df.drop_duplicates().copy()
print(f"\nRow count after dropping exact duplicates: {len(df):,}")

# (b) Clean booking_channel strings
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print(f"Cleaned unique booking_channel categories: {df['booking_channel'].unique().tolist()}")

# (c) Convert date columns
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])

# (d) Drop rows where appt_date or status is missing
df = df.dropna(subset=["appt_date", "status"]).copy()
print(f"Row count after dropping missing appt_date or status: {len(df):,}")


Raw shape: (1020, 14)

Missing values count per column:
appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           1
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              4
dtype: int64

Duplicate rows count: 20
Distinct booking_channel values before cleaning: 16

Row count after dropping exact duplicates: 1,000
Cleaned unique booking_channel categories: ['Phone', 'Walk-In', 'App']
Row count after dropping missing appt_date or status: 990


## 3. Domain Validation, Imputation & Outlier Capping

Clean invalid entries and outliers:
1. Set patient ages below 1 or above 100 to `NaN`.
2. Impute missing `age` and `distance_km` with their respective medians.
3. Compute IQR fences ($Q_1 - 1.5 \times \text{IQR}$, $Q_3 + 1.5 \times \text{IQR}$) for `distance_km` and `fee_inr`, and cap values using `np.clip`.
4. Round `distance_km` to 1 decimal place and `fee_inr` to whole rupees.


In [3]:
# (a) Invalidate impossible ages (< 1 or > 100)
invalid_age_mask = (df["age"] < 1) | (df["age"] > 100)
print(f"Number of impossible ages invalidated: {invalid_age_mask.sum()}")
df.loc[invalid_age_mask, "age"] = np.nan
print(f"Total missing ages to impute: {df['age'].isnull().sum()}")

# (b) Impute missing values using medians
median_age = df["age"].median()
median_distance = df["distance_km"].median()
print(f"Median age: {median_age:.1f}, Median distance: {median_distance:.2f} km")

df["age"] = df["age"].fillna(median_age).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(median_distance)

# (c) IQR outlier calculation and capping
# distance_km
q1_dist = df["distance_km"].quantile(0.25)
q3_dist = df["distance_km"].quantile(0.75)
iqr_dist = q3_dist - q1_dist
uf_dist = q3_dist + 1.5 * iqr_dist
lf_dist = q1_dist - 1.5 * iqr_dist
capped_dist_count = ((df["distance_km"] > uf_dist) | (df["distance_km"] < lf_dist)).sum()
print(f"\ndistance_km: Q1={q1_dist:.3f}, Q3={q3_dist:.3f}, IQR={iqr_dist:.3f}, Upper Fence={uf_dist:.4f}")
print(f"Capped distance_km values: {capped_dist_count}")
df["distance_km"] = np.clip(df["distance_km"], lf_dist, uf_dist)

# fee_inr
q1_fee = df["fee_inr"].quantile(0.25)
q3_fee = df["fee_inr"].quantile(0.75)
iqr_fee = q3_fee - q1_fee
uf_fee = q3_fee + 1.5 * iqr_fee
lf_fee = q1_fee - 1.5 * iqr_fee
capped_fee_count = ((df["fee_inr"] > uf_fee) | (df["fee_inr"] < lf_fee)).sum()
print(f"\nfee_inr: Q1={q1_fee:.1f}, Q3={q3_fee:.1f}, IQR={iqr_fee:.1f}, Upper Fence={uf_fee:.2f}")
print(f"Capped fee_inr values (10x keying errors): {capped_fee_count}")
df["fee_inr"] = np.clip(df["fee_inr"], lf_fee, uf_fee)

# (d) Rounding
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print(f"\nMaximum fee_inr after capping: {df['fee_inr'].max():,}")


Number of impossible ages invalidated: 5
Total missing ages to impute: 17
Median age: 48.0, Median distance: 8.25 km

distance_km: Q1=4.625, Q3=13.000, IQR=8.375, Upper Fence=25.5625
Capped distance_km values: 26

fee_inr: Q1=500.0, Q3=787.5, IQR=287.5, Upper Fence=1218.75
Capped fee_inr values (10x keying errors): 3

Maximum fee_inr after capping: 1,219


## 4. Feature Engineering & Numerical Summary

Derive operational features:
- `lead_days`: Calendar days between `booked_on` and `appt_date`.
- `no_show`: Binary indicator (1 if `status == "No-Show"`, else 0).

Compute vectorized summary statistics using NumPy (column means, share of lead times > 14 days, and distance z-scores).


In [4]:
# Feature engineering
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

print(f"lead_days range: {df['lead_days'].min()} to {df['lead_days'].max()} days")
print(f"Total no-shows: {df['no_show'].sum():,} / {len(df):,} ({df['no_show'].mean()*100:.1f}%)")

# Extract numeric array for NumPy computations
num_cols = ["age", "distance_km", "fee_inr", "lead_days"]
X_num = df[num_cols].to_numpy()

# (a) Column means (axis=0)
col_means = np.mean(X_num, axis=0)
for col, m in zip(num_cols, col_means):
    print(f"Mean {col:12s}: {m:.2f}")

# (b) Boolean mask for lead_days > 14
long_lead_mask = df["lead_days"] > 14
print(f"\nBookings with lead_days > 14: {long_lead_mask.sum():,} ({long_lead_mask.mean()*100:.1f}%)")

# (c) Distance z-scores using population standard deviation (np.std)
dist_arr = df["distance_km"].to_numpy()
z_dist = (dist_arr - np.mean(dist_arr)) / np.std(dist_arr)
extreme_dist_count = (np.abs(z_dist) > 2).sum()
print(f"Bookings with |z| > 2 for distance_km: {extreme_dist_count}")


lead_days range: 1 to 45 days
Total no-shows: 218 / 990 (22.0%)
Mean age         : 48.17
Mean distance_km : 9.50
Mean fee_inr     : 635.71
Mean lead_days   : 11.82

Bookings with lead_days > 14: 296 (29.9%)
Bookings with |z| > 2 for distance_km: 53


## 5. Clinic Enrichment via Open-Meteo Weather API

Query the Open-Meteo Forecast API for each clinic's coordinates to fetch midnight-UTC temperatures (`temp_c`). Includes timeout handling and automatic retry.


In [5]:
import requests

clinics_df = pd.read_csv(CLINICS_PATH)
print(f"Clinics master count: {len(clinics_df)}")

temps = []
for idx, row in clinics_df.iterrows():
    c_id = row["clinic_id"]
    name = row["clinic_name"]
    lat = row["latitude"]
    lon = row["longitude"]
    temp = None
    
    for attempt in range(2):
        try:
            resp = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params={
                    "latitude": lat,
                    "longitude": lon,
                    "hourly": "temperature_2m",
                    "forecast_days": 1
                },
                timeout=10
            )
            if resp.status_code == 200:
                payload = resp.json()
                if "hourly" in payload and "temperature_2m" in payload["hourly"]:
                    temp = payload["hourly"]["temperature_2m"][0]
                    break
        except requests.exceptions.RequestException as err:
            if attempt == 1:
                print(f"Warning: Failed fetching weather for {name}: {err}")
                
    temps.append(temp)
    print(f"Clinic {c_id} ({name:25s}): temp_c = {temp}")

clinics_df["temp_c"] = temps
clinics_df.to_csv("clinics_enriched.csv", index=False)
print("\nSuccessfully saved clinics_enriched.csv")


Clinics master count: 8


Clinic C01 (Kothrud Family Clinic    ): temp_c = 20.9


Clinic C02 (Hadapsar Care Centre     ): temp_c = 21.0


Clinic C03 (Sitabuldi Health Point   ): temp_c = 21.8


Clinic C04 (Malviya Nagar Clinic     ): temp_c = 19.8


Clinic C05 (Vaishali Nagar Clinic    ): temp_c = 19.8


Clinic C06 (Edappally Medical Centre ): temp_c = 25.9


Clinic C07 (Vijay Nagar Clinic       ): temp_c = 20.5


Clinic C08 (Gomti Nagar Care Point   ): temp_c = 19.2

Successfully saved clinics_enriched.csv


## 6. Export Cleaned Dataset & SQL Ingestion Script

Export the verified dataset to `processed/appointments_clean.csv` and generate `sql/load_data.sql` for relational database warehousing.


In [6]:
# Sort by appointment_id
df_clean = df.sort_values("appointment_id").reset_index(drop=True)

# Format dates to standard YYYY-MM-DD text
df_clean["booked_on"] = df_clean["booked_on"].dt.strftime("%Y-%m-%d")
df_clean["appt_date"] = df_clean["appt_date"].dt.strftime("%Y-%m-%d")

target_columns = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status",
    "lead_days", "no_show"
]
df_clean = df_clean[target_columns]

# Validation checks
distinct_ids = df_clean["appointment_id"].nunique()
total_rows = len(df_clean)
total_nulls = df_clean.isnull().sum().sum()

print(f"Clean Table Rows: {total_rows:,}")
print(f"Distinct appointment_id count: {distinct_ids:,}")
print(f"Total null values across all 16 columns: {total_nulls}")
assert distinct_ids == total_rows == 990, "Row count or ID uniqueness check failed!"
assert total_nulls == 0, "Clean table contains unexpected missing values!"

# Export clean appointments
df_clean.to_csv("appointments_clean.csv", index=False)
print("Exported appointments_clean.csv")

# Generate load_data.sql
clinics_enriched = pd.read_csv("clinics_enriched.csv")

with open("load_data.sql", mode="w", encoding="utf-8") as f:
    f.write("-- CareLoop Clinics Data Ingestion Script\n")
    f.write("-- Clinics Master (Parent Table)\n")
    for _, row in clinics_enriched.iterrows():
        temp_val = "NULL" if pd.isna(row["temp_c"]) else f"{row['temp_c']}"
        stmt = f"INSERT INTO clinics VALUES ('{row['clinic_id']}','{row['clinic_name']}','{row['city']}','{row['tier']}',{row['latitude']},{row['longitude']},{temp_val});\n"
        f.write(stmt)
        
    f.write("\n-- Appointments Clean Table (Child Table)\n")
    for _, row in df_clean.iterrows():
        stmt = f"INSERT INTO appointments VALUES ('{row['appointment_id']}','{row['patient_id']}','{row['clinic_id']}','{row['booked_on']}','{row['appt_date']}',{row['age']},'{row['gender']}',{row['distance_km']},{row['fee_inr']},'{row['booking_channel']}','{row['visit_type']}','{row['reminder_sent']}',{row['prior_no_shows']},'{row['status']}',{row['lead_days']},{row['no_show']});\n"
        f.write(stmt)

with open("load_data.sql", "r", encoding="utf-8") as f:
    sql_lines = f.readlines()
insert_stmts = [line for line in sql_lines if line.startswith("INSERT INTO")]
print(f"Generated load_data.sql with {len(insert_stmts)} INSERT statements (8 clinics + 990 appointments).")


Clean Table Rows: 990
Distinct appointment_id count: 990
Total null values across all 16 columns: 0
Exported appointments_clean.csv
Generated load_data.sql with 998 INSERT statements (8 clinics + 990 appointments).
